In [7]:
# nạp các hàm xử lý dữ liệu và đánh giá.

import os
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from IPython.display import display

def find_column(df, names):
    for n in df.columns:
        if n.lower() in [x.lower() for x in names]:
            return n
    return None

def generate_ratings_for_sheet(df, as_float=False, seed=42):
    np.random.seed(seed)
    working = df.copy()
    
    # Đảm bảo có các cột ID
    user_col = find_column(working, ['UserID', 'UserId', 'user_id', 'user'])
    course_col = find_column(working, ['CourseID', 'CourseId', 'course_id', 'course'])
    if user_col is None or course_col is None:
        raise ValueError('Không tìm thấy cột UserID hoặc CourseID trong dữ liệu')

    # Tính toán độ lệch (bias) dựa trên mức độ phổ biến của khóa học
    course_counts = working[course_col].value_counts().astype(float)
    cc_min, cc_max = course_counts.min(), course_counts.max()
    if cc_max - cc_min <= 0:
        norm_counts = course_counts * 0.0
    else:
        norm_counts = (course_counts - cc_min) / (cc_max - cc_min)

    working['_course_pop'] = working[course_col].map(norm_counts).fillna(0.0)

    # Tính toán độ lệch dựa trên mức độ hoạt động của user
    user_counts = working[user_col].value_counts().astype(float)
    uc_min, uc_max = user_counts.min(), user_counts.max()
    if uc_max - uc_min <= 0:
        norm_users = user_counts * 0.0
    else:
        norm_users = (user_counts - uc_min) / (uc_max - uc_min)
    working['_user_act'] = working[user_col].map(norm_users).fillna(0.0)

    # Base mean rating và effect sizes
    base = 3.2
    course_effect = working['_course_pop'] * 1.2  # Tăng tối đa +1.2
    user_effect = (working['_user_act'] - 0.5) * 0.6  # Nằm trong khoảng [-0.3, +0.3]

    raw_means = base + course_effect + user_effect

    # Thêm nhiễu ngẫu nhiên và giới hạn trong khoảng 1..5
    noise = np.random.normal(0, 0.6, size=len(working))
    raw = raw_means + noise
    
    ratings = np.clip(np.round(raw, 2) if as_float else np.rint(raw), 1, 5)

    if as_float:
        working['Rating'] = ratings.astype(float)
    else:
        working['Rating'] = ratings.astype(int)

    # Xóa các cột phụ trợ
    working = working.drop(columns=['_course_pop', '_user_act'])
    return working

def evaluate_with_noisy_prediction(df, rating_col='Rating', noise_std=1.0, threshold=4.0, seed=42):
    np.random.seed(seed + 1)
    if rating_col not in df.columns:
        raise ValueError('Không tìm thấy cột rating_col trong dataframe')
        
    true = df[rating_col].astype(float).to_numpy()
    pred = np.round(np.clip(true + np.random.normal(0, noise_std, size=len(true)), 1.0, 5.0), 2)
    
    true_rel = (true >= threshold).astype(int)
    pred_rel = (pred >= threshold).astype(int)
    
    p, r, f1, _ = precision_recall_fscore_support(true_rel, pred_rel, average='binary', zero_division=0)
    return p, r, f1

In [22]:
# Cấu hình đường dẫn và tham số (thay thế cho argparse)
input_interactions = r'D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions.csv'
output_interactions = r'D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions_with_ratings.csv'

as_float = False
noise_std = 1.0
threshold = 4.0
force = True # Đặt True để ghi đè nếu cột Rating đã tồn tại
seed = 42

if not os.path.exists(input_interactions):
    print(f'Không tìm thấy file: {input_interactions}')
else:
    # 1. Đọc dữ liệu
    df_interactions = pd.read_csv(input_interactions)
    print(f"Đã tải {len(df_interactions)} dòng dữ liệu tương tác.")

    # 2. Tạo hoặc điền Rating
    if 'Rating' in df_interactions.columns and not force:
        print('Cột Rating đã tồn tại. Chuyển force=True để ghi đè.')
    else:
        df_interactions = generate_ratings_for_sheet(df_interactions, as_float=as_float, seed=seed)
        print(f"Đã điền Rating cho {len(df_interactions)} dòng.")

    # 3. Đánh giá mô phỏng
    p, r, f1 = evaluate_with_noisy_prediction(
        df_interactions, 
        rating_col='Rating', 
        noise_std=noise_std, 
        threshold=threshold, 
        seed=seed
    )
    
    print(f'\n--- Kết quả đánh giá (simulated predictions) với threshold {threshold} ---')
    print(f'Precision: {p:.4f}')
    print(f'Recall:    {r:.4f}')
    print(f'F1-score:  {f1:.4f}\n')

    # 4. Lưu lại thành file CSV mới
    df_interactions.to_csv(output_interactions, index=False)
    print(f'Đã lưu file chứa mock ratings tại: {output_interactions}')
    
    # 5. Hiển thị trực tiếp dưới cell
    display(df_interactions.head(10))

Đã tải 184065 dòng dữ liệu tương tác.
Đã điền Rating cho 184065 dòng.

--- Kết quả đánh giá (simulated predictions) với threshold 4.0 ---
Precision: 0.6556
Recall:    0.5350
F1-score:  0.5892

Đã lưu file chứa mock ratings tại: D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions_with_ratings.csv


,UserID,CourseID,Rating
0,3957,583,3
1,928,444,3
2,6935,660,3
3,6582,637,4
4,4460,1131,3
5,7209,107,3
6,716,619,4
7,9948,587,4
8,758,131,3
9,3656,5,4


In [ ]:
# phân loại danh từ và động từ

import spacy

nlp = spacy.load("en_core_web_sm")

text = """
Become a better manager of people. Develop strategies and skills for hiring, managing performance, and rewarding employees.
"""

doc = nlp(text)

nouns = []
verbs = []

for chunk in doc.noun_chunks:
    tokens = [t.text.lower() for t in chunk if t.pos_ not in ["DET", "PRON"]]
    phrase = " ".join(tokens)
    phrase = phrase.replace(" - ", "-")   # sửa khoảng trắng quanh -
    nouns.append(phrase)

for token in doc:
    if token.pos_ == "VERB":
        verbs.append(token.lemma_.lower())

nouns = sorted(set(nouns))
verbs = sorted(set(verbs))

print("NOUN PHRASES:")
print(nouns)

print("\nVERBS:")
print(verbs)

NOUN PHRASES:
['better manager', 'hiring', 'managing performance', 'people', 'rewarding employees', 'skills', 'strategies']

VERBS:
['become', 'develop', 'manage', 'reward']


In [ ]:
# chuyển vào program type sau khi phân loại
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

file_path = r"D:\KLKS\Neo4j\online-course-recommendation-system\Online_Courses_Full_2.csv"

# đọc csv với encoding khác
df = pd.read_csv(file_path)

def extract_keywords(text):
    if pd.isna(text):
        return ""

    doc = nlp(str(text))

    nouns = []
    verbs = []

    for chunk in doc.noun_chunks:
        tokens = [t.text.lower() for t in chunk if t.pos_ not in ["DET", "PRON"]]
        phrase = " ".join(tokens)
        phrase = phrase.replace(" - ", "-")
        if phrase:
            nouns.append(phrase)

    for token in doc:
        if token.pos_ == "VERB":
            verbs.append(token.lemma_.lower())

    keywords = sorted(set(nouns + verbs))

    return ", ".join(keywords)

df["Program Type"] = df["Short Intro"].apply(extract_keywords)

df.to_csv(file_path, index=False)

print("Done!")

Done!


In [ ]:
# nhận diện thực thể và lưu file
import pandas as pd
import spacy

# Tải mô hình tiếng Anh của spaCy
nlp = spacy.load("en_core_web_sm")

file_path = r"D:\KLKS\Neo4j\online-course-recommendation-system\Online_Courses_Full_2.csv"

# Đọc file CSV
df = pd.read_csv(file_path)

def extract_ner(text):
    if pd.isna(text):
        return ""

    doc = nlp(str(text))

    entities = []

    # Danh sách các nhãn bạn muốn giữ lại
    target_labels = ["ORG", "PERSON", "PRODUCT", "LANGUAGE", "WORK_OF_ART"]

    for ent in doc.ents:
        if ent.label_ in target_labels:
            entity_text = ent.text.strip()
            if entity_text:
                entities.append(entity_text)

    # Loại bỏ các thực thể trùng lặp và sắp xếp lại
    unique_entities = sorted(set(entities))

    return ", ".join(unique_entities)

# Áp dụng hàm mới để trích xuất NER
df["Program Type"] = df["Short Intro"].apply(extract_ner)

# Lưu lại file CSV
df.to_csv(file_path, index=False,
    encoding="utf-8-sig")

print("Done!")

d:\Python\Python31011\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Done!


In [28]:
import pandas as pd

courses = pd.read_csv(r"D:\KLKS\Neo4j\online-course-recommendation-system\Online_Courses_Full.csv", encoding='latin1')
ratings = pd.read_csv(r"D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions.csv")

In [ ]:
# 1. CHECK USER ID MISSING
# =========================
user_ids = sorted(ratings['UserID'].dropna().unique())
expected_user_ids = set(range(1, max(user_ids) + 1))
missing_users = sorted(expected_user_ids - set(user_ids))

print("=== USER ID CHECK ===")
if missing_users:
    print(f"Thiếu {len(missing_users)} userID:", missing_users[:20], "..." if len(missing_users) > 20 else "")
else:
    print("Không thiếu userID nào")

=== USER ID CHECK ===
Không thiếu userID nào


In [ ]:
# 2. CHECK COURSE ID MISSING
# =========================
course_ids = sorted(courses['CourseID'].dropna().unique())
expected_course_ids = set(range(1, max(course_ids) + 1))
missing_courses = sorted(expected_course_ids - set(course_ids))

print("\n=== COURSE ID CHECK ===")
if missing_courses:
    print(f"Thiếu {len(missing_courses)} courseID:", missing_courses[:20], "..." if len(missing_courses) > 20 else "")
else:
    print("Không thiếu courseID nào")


=== COURSE ID CHECK ===
Không thiếu courseID nào


In [26]:
# kiểm tra tb rating trong courses khớp với rating
avg_rating_calc = (
    ratings.groupby('CourseID')['Rating']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'Rating': 'calculated_avg'})
)

# Lấy rating đang lưu trong file courses
courses_rating = courses[['CourseID', 'Rating']].copy()
courses_rating['Rating'] = courses_rating['Rating'].round(2)
courses_rating = courses_rating.rename(columns={'Rating': 'stored_avg'})

# Ghép 2 bảng để so sánh
compare = pd.merge(courses_rating, avg_rating_calc, on='CourseID', how='left')

# Tính độ lệch
compare['diff'] = (compare['stored_avg'] - compare['calculated_avg']).abs()

# Lọc course bị lệch
wrong_courses = compare[compare['diff'] > 0.01]

print("\n=== AVG RATING CHECK ===")
if not wrong_courses.empty:
    print(f"Có {len(wrong_courses)} course bị lệch:")
    print(wrong_courses[['CourseID', 'stored_avg', 'calculated_avg', 'diff']].sort_values(by='CourseID'))
else:
    print("Tất cả course đều đúng average rating")


=== AVG RATING CHECK ===
Tất cả course đều đúng average rating


In [ ]:
# kiểm tra courseID bên rating có hợp lý bên trong courses?
invalid_course_ids = sorted(set(ratings['CourseID']) - set(courses['CourseID']))

print("\n=== INVALID COURSE ID IN RATINGS ===")
if invalid_course_ids:
    print("Có CourseID trong ratings nhưng không tồn tại trong courses:")
    print(invalid_course_ids)
else:
    print("Tất cả CourseID trong ratings đều hợp lệ")


=== INVALID COURSE ID IN RATINGS ===
Tất cả CourseID trong ratings đều hợp lệ


In [31]:
# 8. KHÓA HỌC CHƯA TỪNG ĐƯỢC USER TƯƠNG TÁC
# =========================
ratings_dedup = ratings.drop_duplicates(
    subset=["UserID", "CourseID"],
    keep="last"
)

interacted_course_ids = set(ratings_dedup["CourseID"].unique())

uninteracted_courses = courses[~courses["CourseID"].isin(interacted_course_ids)].copy()
uninteracted_courses = uninteracted_courses.sort_values(by="CourseID")

print("\n=== CÁC KHÓA HỌC CHƯA TỪNG ĐƯỢC USER TƯƠNG TÁC ===")
print(f"Số lượng course chưa được tương tác: {len(uninteracted_courses)}")

if len(uninteracted_courses) > 0:
    print(uninteracted_courses[["CourseID", "Title", "Category"]].to_string(index=False))
else:
    print("Tất cả khóa học đều đã có ít nhất 1 tương tác")

# =========================
# 9. KIỂM TRA COURSEID TRONG RATINGS NHƯNG KHÔNG TỒN TẠI TRONG COURSES
# =========================
invalid_course_ids = sorted(set(ratings_dedup["CourseID"]) - set(courses["CourseID"]))

print("\n=== INVALID COURSE ID IN RATINGS ===")
if invalid_course_ids:
    print("Có CourseID trong ratings nhưng không tồn tại trong courses:")
    print(invalid_course_ids)
else:
    print("Tất cả CourseID trong ratings đều hợp lệ")


=== CÁC KHÓA HỌC CHƯA TỪNG ĐƯỢC USER TƯƠNG TÁC ===
Số lượng course chưa được tương tác: 98
 CourseID                                                                             Title                         Category
       68            Data Engineering, Big Data, and Machine Learning on GCP Specialization                     Data Science
       89             Java Programming and Software Engineering Fundamentals Specialization                 Computer Science
      104                                          Mergers and Acquisitions  Specialization                         Business
      115           Food Sustainability, Mindful Eating, and Healthy Cooking Specialization                           Health
      119                                         Clinical Trials Operations Specialization                           Health
      246                                          Digital Signal Processing Specialization Physical Science and Engineering
      256          Qualitative Re

In [33]:
# CHECK NUMBER OF VIEWERS
# =========================

# 1. Dedup (mỗi user chỉ tính 1 lần / course)
ratings_dedup = ratings.drop_duplicates(
    subset=["UserID", "CourseID"],
    keep="last"
)

# 2. Tính số user unique cho mỗi course
viewer_calc = (
    ratings_dedup.groupby("CourseID")["UserID"]
    .nunique()
    .reset_index()
    .rename(columns={"UserID": "calculated_viewers"})
)

# 3. Lấy số viewers từ courses
course_viewers = courses[["CourseID", "Number of viewers"]].copy()
course_viewers = course_viewers.rename(columns={"Number of viewers": "stored_viewers"})

# 4. Merge để so sánh
compare_viewers = pd.merge(course_viewers, viewer_calc, on="CourseID", how="left")

# Nếu course chưa có ai xem → fill 0
compare_viewers["calculated_viewers"] = compare_viewers["calculated_viewers"].fillna(0).astype(int)

# 5. So sánh lệch
compare_viewers["diff"] = compare_viewers["stored_viewers"] - compare_viewers["calculated_viewers"]

wrong_viewers = compare_viewers[compare_viewers["diff"] != 0]

# 6. In kết quả
print("\n=== NUMBER OF VIEWERS CHECK ===")
if not wrong_viewers.empty:
    print(f"Có {len(wrong_viewers)} course bị lệch viewers:")
    print(
        wrong_viewers[["CourseID", "stored_viewers", "calculated_viewers", "diff"]]
        .sort_values(by="CourseID")
        .to_string(index=False)
    )
else:
    print("Tất cả course đều đúng Number of viewers")


=== NUMBER OF VIEWERS CHECK ===
Có 1 course bị lệch viewers:
 CourseID  stored_viewers  calculated_viewers  diff
       65               0                   1    -1
